# 16 - Clustering e Correlação no Brasileirão

Análise de **perfis de times** usando K-Means e correlação entre variáveis de desempenho.

- Clustering por métricas de performance (gols, aproveitamento, casa/fora)
- Correlação entre variáveis e posição final
- Identificação de perfis: candidatos ao título, meio de tabela, ameaçados de queda

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy import stats as sp_stats

df = pd.read_csv('../data/raw/campeonato-brasileiro-full.csv')
df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')
df['ano'] = df['data'].dt.year
df = df.dropna(subset=['rodata'])
df['rodata'] = df['rodata'].astype(int)

# Stats (disponível 2003-2024)
df_stats = pd.read_csv('../data/raw/campeonato-brasileiro-estatisticas-full.csv')

# Foco em temporadas completas recentes
ANOS = list(range(2006, 2026))  # 20 times por temporada
df_era = df[df['ano'].isin(ANOS)].copy()
print(f'Temporadas analisadas: {min(ANOS)}-{max(ANOS)}')
print(f'Jogos: {len(df_era)}')

Temporadas analisadas: 2006-2025
Jogos: 7599


In [2]:
# === CALCULAR MÉTRICAS POR TIME/TEMPORADA ===

def calcular_metricas_temporada(df_era):
    """Calcula métricas detalhadas por time em cada temporada."""
    registros = []
    
    for ano in df_era['ano'].unique():
        df_ano = df_era[df_era['ano'] == ano]
        if len(df_ano) < 300:  # temporada incompleta
            continue
        
        times = sorted(set(df_ano['mandante'].unique()) | set(df_ano['visitante'].unique()))
        
        for time in times:
            mand = df_ano[df_ano['mandante'] == time]
            visit = df_ano[df_ano['visitante'] == time]
            
            # Pontos e resultados
            pts = 0
            vit, emp, der = 0, 0, 0
            vit_m, vit_v = 0, 0
            gm_total, gs_total = 0, 0
            gm_casa, gs_casa, gm_fora, gs_fora = 0, 0, 0, 0
            
            for _, j in mand.iterrows():
                gm, gv = int(j['mandante_Placar']), int(j['visitante_Placar'])
                gm_total += gm; gs_total += gv
                gm_casa += gm; gs_casa += gv
                if gm > gv: pts += 3; vit += 1; vit_m += 1
                elif gm == gv: pts += 1; emp += 1
                else: der += 1
            
            for _, j in visit.iterrows():
                gm, gv = int(j['mandante_Placar']), int(j['visitante_Placar'])
                gm_total += gv; gs_total += gm
                gm_fora += gv; gs_fora += gm
                if gv > gm: pts += 3; vit += 1; vit_v += 1
                elif gm == gv: pts += 1; emp += 1
                else: der += 1
            
            jogos = len(mand) + len(visit)
            if jogos == 0:
                continue
            
            registros.append({
                'ano': ano, 'time': time,
                'pontos': pts, 'jogos': jogos,
                'vitorias': vit, 'empates': emp, 'derrotas': der,
                'gols_marcados': gm_total, 'gols_sofridos': gs_total,
                'saldo_gols': gm_total - gs_total,
                'aproveitamento': pts / (jogos * 3) * 100,
                'media_gm': gm_total / jogos,
                'media_gs': gs_total / jogos,
                'gm_casa': gm_casa / max(len(mand), 1),
                'gs_casa': gs_casa / max(len(mand), 1),
                'gm_fora': gm_fora / max(len(visit), 1),
                'gs_fora': gs_fora / max(len(visit), 1),
                'vit_casa': vit_m, 'vit_fora': vit_v,
                'aprov_casa': (vit_m * 3 + (len(mand) - vit_m - sum(1 for _, j in mand.iterrows() if int(j['mandante_Placar']) < int(j['visitante_Placar'])))) / max(len(mand) * 3, 1) * 100 if len(mand) > 0 else 0,
            })
    
    return pd.DataFrame(registros)

df_metricas = calcular_metricas_temporada(df_era)

# Calcular posição final por temporada
for ano in df_metricas['ano'].unique():
    mask = df_metricas['ano'] == ano
    df_metricas.loc[mask, 'posicao'] = df_metricas.loc[mask].sort_values(
        ['pontos', 'vitorias', 'saldo_gols', 'gols_marcados'], ascending=False
    ).reset_index().reset_index()['level_0'].values + 1

df_metricas['posicao'] = df_metricas['posicao'].astype(int)
print(f'Registros time/temporada: {len(df_metricas)}')
print(f'Colunas: {list(df_metricas.columns)}')
df_metricas.head()

Registros time/temporada: 384
Colunas: ['ano', 'time', 'pontos', 'jogos', 'vitorias', 'empates', 'derrotas', 'gols_marcados', 'gols_sofridos', 'saldo_gols', 'aproveitamento', 'media_gm', 'media_gs', 'gm_casa', 'gs_casa', 'gm_fora', 'gs_fora', 'vit_casa', 'vit_fora', 'aprov_casa', 'posicao']


,ano,time,pontos,jogos,vitorias,empates,derrotas,gols_marcados,gols_sofridos,saldo_gols,...,media_gm,media_gs,gm_casa,gs_casa,gm_fora,gs_fora,vit_casa,vit_fora,aprov_casa,posicao
0,2006,Athletico-PR,48,38,13,9,16,61,62,-1,...,1.605263,1.631579,1.736842,1.526316,1.473684,1.736842,9,4,52.631579,1
1,2006,Botafogo-RJ,51,38,13,12,13,52,50,2,...,1.368421,1.315789,1.842105,1.368421,0.894737,1.263158,10,3,59.649123,2
2,2006,Corinthians,53,38,15,8,15,41,46,-5,...,1.078947,1.210526,1.157895,1.210526,1.000000,1.210526,9,6,52.631579,3
3,2006,Cruzeiro,53,38,14,11,13,52,45,7,...,1.368421,1.184211,2.000000,1.052632,0.736842,1.315789,10,4,66.666667,4
4,2006,Figueirense,57,38,15,12,11,52,44,8,...,1.368421,1.157895,1.315789,0.789474,1.421053,1.526316,9,6,59.649123,5


In [3]:
# === CORREÇÃO: Calcular posição corretamente ===

posicoes = []
for ano in df_metricas['ano'].unique():
    temp = df_metricas[df_metricas['ano'] == ano].sort_values(
        ['pontos', 'vitorias', 'saldo_gols', 'gols_marcados'], ascending=False
    ).reset_index()
    temp['posicao'] = range(1, len(temp) + 1)
    posicoes.append(temp[['ano', 'time', 'posicao']])

df_pos = pd.concat(posicoes)
df_metricas = df_metricas.drop(columns=['posicao']).merge(df_pos, on=['ano', 'time'])

# Flags de zona
df_metricas['zona'] = df_metricas['posicao'].apply(
    lambda p: 'Título' if p == 1 else ('G4' if p <= 4 else ('G6' if p <= 6 else ('Z4' if p >= 17 else 'Meio')))
)

print('Distribuição por zona:')
print(df_metricas['zona'].value_counts())

Distribuição por zona:
zona
Meio      190
Z4         80
G4         57
G6         38
Título     19
Name: count, dtype: int64


In [4]:
# === GRÁFICO 1: MATRIZ DE CORRELAÇÃO ===

cols_corr = ['pontos', 'aproveitamento', 'gols_marcados', 'gols_sofridos',
             'saldo_gols', 'media_gm', 'media_gs', 'gm_casa', 'gs_casa',
             'gm_fora', 'gs_fora', 'vitorias', 'empates', 'derrotas',
             'vit_casa', 'vit_fora', 'posicao']

corr = df_metricas[cols_corr].corr()

# Labels mais amigáveis
labels = {
    'pontos': 'Pontos', 'aproveitamento': 'Aproveitamento',
    'gols_marcados': 'Gols Marcados', 'gols_sofridos': 'Gols Sofridos',
    'saldo_gols': 'Saldo Gols', 'media_gm': 'Média GM/jogo',
    'media_gs': 'Média GS/jogo', 'gm_casa': 'GM Casa/jogo',
    'gs_casa': 'GS Casa/jogo', 'gm_fora': 'GM Fora/jogo',
    'gs_fora': 'GS Fora/jogo', 'vitorias': 'Vitórias',
    'empates': 'Empates', 'derrotas': 'Derrotas',
    'vit_casa': 'Vitórias Casa', 'vit_fora': 'Vitórias Fora',
    'posicao': 'Posição Final'
}
nice_labels = [labels.get(c, c) for c in cols_corr]

fig_corr = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=nice_labels, y=nice_labels,
    colorscale='RdBu_r',
    zmid=0, zmin=-1, zmax=1,
    text=np.round(corr.values, 2),
    texttemplate='%{text}',
    textfont={'size': 9},
    hovertemplate='%{x} vs %{y}<br>Correlação: %{z:.3f}<extra></extra>',
    colorbar=dict(title='r')
))

fig_corr.update_layout(
    title='Matriz de Correlação — Métricas do Brasileirão (2006-2025)<br>'
          '<sup>Correlação de Pearson entre variáveis de desempenho por temporada</sup>',
    width=900, height=800,
    template='plotly_white',
    xaxis_tickangle=-45,
)

fig_corr.show()
fig_corr.write_html('../charts/correlacao_heatmap.html', include_plotlyjs='cdn')
print('Gráfico exportado: charts/correlacao_heatmap.html')

Gráfico exportado: charts/correlacao_heatmap.html


In [5]:
# === GRÁFICO 2: CORRELAÇÃO COM POSIÇÃO FINAL ===
# Quais variáveis mais impactam a posição?

corr_pos = df_metricas[cols_corr].corr()['posicao'].drop('posicao').sort_values()

colors = ['#238636' if v < 0 else '#da3633' for v in corr_pos.values]

fig_cp = go.Figure(go.Bar(
    x=corr_pos.values,
    y=[labels.get(c, c) for c in corr_pos.index],
    orientation='h',
    marker_color=colors,
    text=[f'{v:.3f}' for v in corr_pos.values],
    textposition='outside',
    hovertemplate='%{y}: r = %{x:.3f}<extra></extra>'
))

fig_cp.update_layout(
    title='Correlação com Posição Final — Brasileirão (2006-2025)<br>'
          '<sup>Verde = melhora posição (negativo) | Vermelho = piora posição (positivo)</sup>',
    xaxis_title='Correlação de Pearson (r)',
    template='plotly_white',
    width=800, height=600,
    xaxis=dict(range=[-1.05, 1.05]),
)
fig_cp.add_vline(x=0, line_color='gray', line_dash='dash')

fig_cp.show()
fig_cp.write_html('../charts/correlacao_posicao.html', include_plotlyjs='cdn')
print('Gráfico exportado: charts/correlacao_posicao.html')

Gráfico exportado: charts/correlacao_posicao.html


In [6]:
# === K-MEANS CLUSTERING ===
# Agrupar times/temporadas por perfil de desempenho

features = ['aproveitamento', 'media_gm', 'media_gs', 'gm_casa', 'gs_casa',
            'gm_fora', 'gs_fora', 'vit_casa', 'vit_fora', 'saldo_gols']

X = df_metricas[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Encontrar K ideal com método do cotovelo
inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig_elbow = go.Figure(go.Scatter(
    x=list(K_range), y=inertias,
    mode='lines+markers', marker=dict(size=10),
    line=dict(color='#58a6ff')
))
fig_elbow.update_layout(
    title='Método do Cotovelo — K ideal para Clustering',
    xaxis_title='Número de Clusters (K)',
    yaxis_title='Inércia',
    template='plotly_white', width=600, height=400
)
fig_elbow.show()

# Usar K=4 (título, G4/G6, meio tabela, Z4)
K = 4
km = KMeans(n_clusters=K, n_init=20, random_state=42)
df_metricas['cluster'] = km.fit_predict(X_scaled)

# Ordenar clusters por aproveitamento médio
cluster_order = df_metricas.groupby('cluster')['aproveitamento'].mean().sort_values(ascending=False)
cluster_map = {old: new for new, old in enumerate(cluster_order.index)}
df_metricas['cluster'] = df_metricas['cluster'].map(cluster_map)

cluster_names = {0: 'Candidatos ao Título', 1: 'Libertadores', 2: 'Meio de Tabela', 3: 'Ameaçados/Z4'}
df_metricas['perfil'] = df_metricas['cluster'].map(cluster_names)

print('Perfis identificados:')
for c in sorted(cluster_names.keys()):
    sub = df_metricas[df_metricas['cluster'] == c]
    print(f'  {cluster_names[c]}: {len(sub)} registros, '
          f'aprov. médio={sub["aproveitamento"].mean():.1f}%, '
          f'posição média={sub["posicao"].mean():.1f}')

Perfis identificados:
  Candidatos ao Título: 90 registros, aprov. médio=59.5%, posição média=3.2
  Libertadores: 105 registros, aprov. médio=44.7%, posição média=11.0
  Meio de Tabela: 125 registros, aprov. médio=43.8%, posição média=11.4
  Ameaçados/Z4: 64 registros, aprov. médio=29.8%, posição média=18.9


In [7]:
# === GRÁFICO 3: SCATTER — GOLS MARCADOS vs SOFRIDOS com CLUSTERS ===

cluster_colors = {
    'Candidatos ao Título': '#ffd700',
    'Libertadores': '#238636',
    'Meio de Tabela': '#58a6ff',
    'Ameaçados/Z4': '#da3633'
}

fig_scatter = go.Figure()

for perfil, cor in cluster_colors.items():
    sub = df_metricas[df_metricas['perfil'] == perfil]
    fig_scatter.add_trace(go.Scatter(
        x=sub['media_gm'], y=sub['media_gs'],
        mode='markers',
        marker=dict(color=cor, size=7, opacity=0.6, line=dict(width=0.5, color='white')),
        name=perfil,
        hovertemplate='%{customdata[0]} (%{customdata[1]})<br>'
                      'GM/jogo: %{x:.2f}<br>GS/jogo: %{y:.2f}<br>'
                      'Posição: %{customdata[2]}°<extra>' + perfil + '</extra>',
        customdata=sub[['time', 'ano', 'posicao']].values
    ))

# Diagonal de equilíbrio
fig_scatter.add_trace(go.Scatter(
    x=[0.5, 2.5], y=[0.5, 2.5],
    mode='lines', line=dict(color='gray', dash='dash', width=1),
    showlegend=False, hoverinfo='skip'
))

fig_scatter.add_annotation(x=2.0, y=0.7, text='Ataque forte<br>Defesa forte',
                           showarrow=False, font=dict(color='green', size=10))
fig_scatter.add_annotation(x=0.7, y=2.0, text='Ataque fraco<br>Defesa fraca',
                           showarrow=False, font=dict(color='red', size=10))

fig_scatter.update_layout(
    title='Perfis de Times — Gols Marcados vs Sofridos por Jogo<br>'
          '<sup>K-Means Clustering (K=4) | Brasileirão 2006-2025</sup>',
    xaxis_title='Gols Marcados / Jogo',
    yaxis_title='Gols Sofridos / Jogo',
    template='plotly_white',
    width=900, height=700,
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)')
)

fig_scatter.show()
fig_scatter.write_html('../charts/cluster_scatter.html', include_plotlyjs='cdn')
print('Gráfico exportado: charts/cluster_scatter.html')

Gráfico exportado: charts/cluster_scatter.html


In [8]:
# === GRÁFICO 4: RADAR POR CLUSTER ===

radar_features = ['aproveitamento', 'media_gm', 'gm_casa', 'gm_fora',
                  'vit_casa', 'vit_fora']
radar_labels = ['Aproveitamento', 'Gols/jogo', 'GM Casa', 'GM Fora',
                'Vitórias Casa', 'Vitórias Fora']

# Normalizar para 0-1
radar_data = df_metricas.groupby('perfil')[radar_features].mean()
radar_norm = (radar_data - radar_data.min()) / (radar_data.max() - radar_data.min())

fig_radar = go.Figure()

for perfil in ['Candidatos ao Título', 'Libertadores', 'Meio de Tabela', 'Ameaçados/Z4']:
    if perfil in radar_norm.index:
        vals = radar_norm.loc[perfil].values.tolist()
        vals.append(vals[0])  # fechar o polígono
        fig_radar.add_trace(go.Scatterpolar(
            r=vals,
            theta=radar_labels + [radar_labels[0]],
            name=perfil,
            line=dict(color=cluster_colors[perfil], width=2),
            fill='toself', opacity=0.3
        ))

fig_radar.update_layout(
    title='Radar de Perfil por Cluster — Brasileirão (2006-2025)',
    polar=dict(radialaxis=dict(visible=True, range=[0, 1.05])),
    template='plotly_white',
    width=700, height=600,
)

fig_radar.show()
fig_radar.write_html('../charts/cluster_radar.html', include_plotlyjs='cdn')
print('Gráfico exportado: charts/cluster_radar.html')

Gráfico exportado: charts/cluster_radar.html


In [9]:
# === GRÁFICO 5: SCATTER — APROVEITAMENTO CASA vs FORA com CLUSTERS ===

# Calcular aproveitamento fora de casa
# (precisamos recalcular isso de forma limpa)
for idx, row in df_metricas.iterrows():
    ano, time = row['ano'], row['time']
    df_ano = df_era[df_era['ano'] == ano]
    visit = df_ano[df_ano['visitante'] == time]
    pts_fora = 0
    for _, j in visit.iterrows():
        gm, gv = int(j['mandante_Placar']), int(j['visitante_Placar'])
        if gv > gm: pts_fora += 3
        elif gm == gv: pts_fora += 1
    df_metricas.loc[idx, 'aprov_fora'] = pts_fora / max(len(visit) * 3, 1) * 100
    
    mand = df_ano[df_ano['mandante'] == time]
    pts_casa = 0
    for _, j in mand.iterrows():
        gm, gv = int(j['mandante_Placar']), int(j['visitante_Placar'])
        if gm > gv: pts_casa += 3
        elif gm == gv: pts_casa += 1
    df_metricas.loc[idx, 'aprov_casa'] = pts_casa / max(len(mand) * 3, 1) * 100

fig_cv = go.Figure()

for perfil, cor in cluster_colors.items():
    sub = df_metricas[df_metricas['perfil'] == perfil]
    fig_cv.add_trace(go.Scatter(
        x=sub['aprov_casa'], y=sub['aprov_fora'],
        mode='markers',
        marker=dict(color=cor, size=7, opacity=0.6),
        name=perfil,
        hovertemplate='%{customdata[0]} (%{customdata[1]})<br>'
                      'Aprov. Casa: %{x:.1f}%<br>Aprov. Fora: %{y:.1f}%<br>'
                      'Posição: %{customdata[2]}°<extra></extra>',
        customdata=sub[['time', 'ano', 'posicao']].values
    ))

fig_cv.update_layout(
    title='Aproveitamento Casa vs Fora — Perfis de Time<br>'
          '<sup>Cada ponto = um time em uma temporada | K-Means K=4</sup>',
    xaxis_title='Aproveitamento em Casa (%)',
    yaxis_title='Aproveitamento Fora (%)',
    template='plotly_white',
    width=800, height=650,
)

fig_cv.show()
fig_cv.write_html('../charts/cluster_casa_fora.html', include_plotlyjs='cdn')
print('Gráfico exportado: charts/cluster_casa_fora.html')

Gráfico exportado: charts/cluster_casa_fora.html


In [10]:
# === GRÁFICO 6: HEATMAP — FREQUÊNCIA DE CADA TIME EM CADA CLUSTER ===
# Quais times historicamente aparecem mais em cada perfil?

# Top 20 times com mais temporadas
top_times = df_metricas['time'].value_counts().head(20).index.tolist()
df_top = df_metricas[df_metricas['time'].isin(top_times)]

# Matriz: time x perfil → contagem
ct = pd.crosstab(df_top['time'], df_top['perfil'])
# Ordenar por proporção em 'Candidatos ao Título'
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ordem = ct_pct.reindex(columns=['Candidatos ao Título', 'Libertadores', 'Meio de Tabela', 'Ameaçados/Z4']).fillna(0)
ordem = ordem.sort_values('Candidatos ao Título', ascending=True).index.tolist()

cols_perfil = ['Candidatos ao Título', 'Libertadores', 'Meio de Tabela', 'Ameaçados/Z4']
ct_pct = ct_pct.reindex(index=ordem, columns=cols_perfil).fillna(0)

fig_heat = go.Figure(data=go.Heatmap(
    z=ct_pct.values,
    x=cols_perfil,
    y=ct_pct.index.tolist(),
    colorscale=[
        [0, '#0d1117'],
        [0.25, '#1a3a5c'],
        [0.5, '#58a6ff'],
        [0.75, '#ffd700'],
        [1.0, '#ff4444']
    ],
    text=np.round(ct_pct.values, 1),
    texttemplate='%{text}%',
    textfont={'size': 10},
    hovertemplate='%{y}<br>%{x}: %{z:.1f}%<extra></extra>',
    colorbar=dict(title='% das temporadas')
))

fig_heat.update_layout(
    title='DNA dos Clubes — Em qual perfil cada time mais aparece?<br>'
          '<sup>% das temporadas em cada cluster | Brasileirão 2006-2025</sup>',
    template='plotly_white',
    width=800, height=700,
)

fig_heat.show()
fig_heat.write_html('../charts/cluster_times_heatmap.html', include_plotlyjs='cdn')
print('Gráfico exportado: charts/cluster_times_heatmap.html')

Gráfico exportado: charts/cluster_times_heatmap.html


In [11]:
# === GRÁFICO 7: SCATTER CORRELAÇÕES CHAVE ===
# Saldo de gols vs Posição Final (a correlação mais forte)

fig_sg = px.scatter(
    df_metricas, x='saldo_gols', y='posicao',
    color='perfil',
    color_discrete_map=cluster_colors,
    hover_data=['time', 'ano', 'pontos'],
    labels={'saldo_gols': 'Saldo de Gols', 'posicao': 'Posição Final',
            'perfil': 'Perfil'},
    title='Saldo de Gols vs Posição Final<br>'
          '<sup>Colorido por cluster | Brasileirão 2006-2025</sup>',
    width=800, height=600,
    template='plotly_white'
)

# Linha de tendência
slope, intercept, r_value, _, _ = sp_stats.linregress(df_metricas['saldo_gols'], df_metricas['posicao'])
x_range = np.linspace(df_metricas['saldo_gols'].min(), df_metricas['saldo_gols'].max(), 100)
fig_sg.add_trace(go.Scatter(
    x=x_range, y=slope * x_range + intercept,
    mode='lines', line=dict(color='gray', dash='dash'),
    name=f'Tendência (r={r_value:.3f})', showlegend=True
))

fig_sg.update_yaxes(autorange='reversed')
fig_sg.show()
fig_sg.write_html('../charts/correlacao_saldo_posicao.html', include_plotlyjs='cdn')
print(f'r = {r_value:.3f} (p < 0.001)')
print('Gráfico exportado: charts/correlacao_saldo_posicao.html')

r = -0.883 (p < 0.001)
Gráfico exportado: charts/correlacao_saldo_posicao.html


In [12]:
# === CLASSIFICAÇÃO 2026 — EM QUAL CLUSTER CADA TIME ESTÁ? ===

# Dados atuais de 2026 (rodada 6)
df_2026 = df[df['ano'] == 2026].copy()
if len(df_2026) > 0:
    metricas_2026 = []
    times_2026 = sorted(set(df_2026['mandante'].unique()) | set(df_2026['visitante'].unique()))
    
    for time in times_2026:
        mand = df_2026[df_2026['mandante'] == time]
        visit = df_2026[df_2026['visitante'] == time]
        jogos = len(mand) + len(visit)
        if jogos == 0:
            continue
        
        pts, vit, gm, gs = 0, 0, 0, 0
        gm_c, gs_c, gm_f, gs_f = 0, 0, 0, 0
        vit_c, vit_f = 0, 0
        
        for _, j in mand.iterrows():
            gm_j, gv_j = int(j['mandante_Placar']), int(j['visitante_Placar'])
            gm += gm_j; gs += gv_j; gm_c += gm_j; gs_c += gv_j
            if gm_j > gv_j: pts += 3; vit += 1; vit_c += 1
            elif gm_j == gv_j: pts += 1
        
        for _, j in visit.iterrows():
            gm_j, gv_j = int(j['mandante_Placar']), int(j['visitante_Placar'])
            gm += gv_j; gs += gm_j; gm_f += gv_j; gs_f += gm_j
            if gv_j > gm_j: pts += 3; vit += 1; vit_f += 1
            elif gm_j == gv_j: pts += 1
        
        metricas_2026.append({
            'time': time,
            'aproveitamento': pts / (jogos * 3) * 100,
            'media_gm': gm / jogos,
            'media_gs': gs / jogos,
            'gm_casa': gm_c / max(len(mand), 1),
            'gs_casa': gs_c / max(len(mand), 1),
            'gm_fora': gm_f / max(len(visit), 1),
            'gs_fora': gs_f / max(len(visit), 1),
            'vit_casa': vit_c * (19 / max(len(mand), 1)),  # projetar para 19 jogos
            'vit_fora': vit_f * (19 / max(len(visit), 1)),
            'saldo_gols': (gm - gs) * (38 / jogos),  # projetar para 38 jogos
            'pontos': pts,
        })
    
    df_2026_m = pd.DataFrame(metricas_2026)
    
    # Aplicar o mesmo scaler e modelo
    X_2026 = scaler.transform(df_2026_m[features])
    clusters_2026 = km.predict(X_2026)
    # Aplicar mesmo mapeamento
    df_2026_m['cluster'] = [cluster_map.get(c, c) for c in clusters_2026]
    df_2026_m['perfil'] = df_2026_m['cluster'].map(cluster_names)
    
    print('Perfis dos times em 2026 (baseado nas 6 rodadas):')
    print(f'{"":-<60}')
    for perfil in ['Candidatos ao Título', 'Libertadores', 'Meio de Tabela', 'Ameaçados/Z4']:
        sub = df_2026_m[df_2026_m['perfil'] == perfil]
        if len(sub) > 0:
            times_str = ', '.join(sub.sort_values('aproveitamento', ascending=False)['time'].tolist())
            print(f'\n{perfil}:')
            print(f'  {times_str}')

Perfis dos times em 2026 (baseado nas 6 rodadas):
------------------------------------------------------------

Candidatos ao Título:
  São Paulo, Fluminense, Palmeiras, Athletico-PR, Flamengo, Coritiba, Corinthians, Mirassol

Libertadores:
  Vitória, Grêmio, Botafogo

Meio de Tabela:
  Bragantino, Bahia, Atlético-MG, Internacional

Ameaçados/Z4:
  Chapecoense, Vasco, Santos, Cruzeiro, Remo


In [13]:
# === RESUMO ===

print('=' * 60)
print('CLUSTERING E CORRELAÇÃO — BRASILEIRÃO')
print('=' * 60)

print('\nPrincipais correlações com posição final:')
for var, r in corr_pos.items():
    if abs(r) > 0.5:
        direcao = '↑ melhor' if r < 0 else '↓ pior'
        print(f'  {labels.get(var, var):<20s} r={r:+.3f} ({direcao})')

print(f'\n4 perfis identificados via K-Means:')
for c in sorted(cluster_names.keys()):
    sub = df_metricas[df_metricas['cluster'] == c]
    print(f'  {cluster_names[c]:<25s} — aprov. {sub["aproveitamento"].mean():.1f}%, '
          f'posição média {sub["posicao"].mean():.1f}°')

print(f'\n7 gráficos exportados para charts/')

CLUSTERING E CORRELAÇÃO — BRASILEIRÃO

Principais correlações com posição final:
  Aproveitamento       r=-0.943 (↑ melhor)
  Vitórias             r=-0.924 (↑ melhor)
  Pontos               r=-0.922 (↑ melhor)
  Saldo Gols           r=-0.883 (↑ melhor)
  Vitórias Casa        r=-0.810 (↑ melhor)
  Vitórias Fora        r=-0.755 (↑ melhor)
  Gols Marcados        r=-0.724 (↑ melhor)
  Média GM/jogo        r=-0.715 (↑ melhor)
  GM Casa/jogo         r=-0.632 (↑ melhor)
  GM Fora/jogo         r=-0.544 (↑ melhor)
  GS Casa/jogo         r=+0.571 (↓ pior)
  Gols Sofridos        r=+0.591 (↓ pior)
  GS Fora/jogo         r=+0.626 (↓ pior)
  Média GS/jogo        r=+0.709 (↓ pior)
  Derrotas             r=+0.773 (↓ pior)

4 perfis identificados via K-Means:
  Candidatos ao Título      — aprov. 59.5%, posição média 3.2°
  Libertadores              — aprov. 44.7%, posição média 11.0°
  Meio de Tabela            — aprov. 43.8%, posição média 11.4°
  Ameaçados/Z4              — aprov. 29.8%, posição médi